# Modern RAG Agent - Complete Demo

This notebook demonstrates the full capabilities of the RAG system:

1. Setting up the RAG agent with advanced features
2. Loading and processing documents (with automatic OCR)
3. Creating a vector database
4. Basic and advanced querying
5. Conversation memory and multi-turn dialogues
6. Advanced retrieval techniques (reranking, hybrid search, MMR)
7. Evaluation and benchmarking


## Step 1: Install and Import Dependencies


In [ ]:
import os
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Add project root to path
sys.path.insert(0, str(Path(__file__).parent.parent))

from src import RAGAgent, DocumentLoader, ConversationMemory, AdvancedRetriever, RAGConfig

print("✅ Imports successful!")
print("📦 Available modules:")
print("   - RAGAgent: Main RAG agent")
print("   - DocumentLoader: Document processing")
print("   - ConversationMemory: Multi-turn dialogue")
print("   - AdvancedRetriever: Reranking, hybrid search, MMR")
print("   - RAGConfig: Configuration management")


✅ Imports successful!


## Step 2: Check Ollama Setup


In [2]:
import subprocess
import requests

def check_ollama():
    """Check if Ollama is installed and running"""
    try:
        # Check if Ollama command exists
        result = subprocess.run(
            ["ollama", "--version"],
            capture_output=True,
            text=True,
            timeout=5
        )
        if result.returncode == 0:
            print(f"✅ Ollama installed: {result.stdout.strip()}")
            
            # Check if service is running
            try:
                response = requests.get("http://localhost:11434/api/tags", timeout=2)
                if response.status_code == 200:
                    models = response.json().get("models", [])
                    if models:
                        print(f"✅ Ollama service running")
                        print(f"📦 Available models: {', '.join([m.get('name', '') for m in models])}")
                        return True, [m.get('name', '') for m in models]
                    else:
                        print("⚠️  No models found. Run: ollama pull llama2")
                        return False, []
                else:
                    print("❌ Ollama service not responding")
                    return False, []
            except Exception as e:
                print(f"❌ Cannot connect to Ollama: {e}")
                print("   Make sure Ollama is running")
                return False, []
        else:
            print("❌ Ollama not found")
            return False, []
    except FileNotFoundError:
        print("❌ Ollama not installed")
        print("   Install from: https://ollama.ai/")
        return False, []
    except Exception as e:
        print(f"❌ Error: {e}")
        return False, []

ollama_ok, available_models = check_ollama()

if ollama_ok and available_models:
    model_name = available_models[0]  # Use first available model
    print(f"\n🎯 Using model: {model_name}")
else:
    model_name = "llama2"  # Default
    print(f"\n⚠️  Will try to use: {model_name} (make sure it's pulled)")


✅ Ollama installed: ollama version is 0.13.0
⚠️  No models found. Run: ollama pull llama2

⚠️  Will try to use: llama2 (make sure it's pulled)


## Step 3: Initialize RAG Agent


In [ ]:
# Initialize the RAG agent with advanced features
print("🚀 Initializing RAG Agent with advanced features...")
print("\nFeatures enabled:")
print("   ✅ Conversation Memory")
print("   ✅ Advanced Retrieval (optional)")

agent = RAGAgent(
    model_name=model_name,
    enable_memory=True,        # Enable conversation memory
    enable_reranking=False,    # Set to True to enable reranking
    enable_hybrid=False,      # Set to True to enable hybrid search
    enable_mmr=False,         # Set to True to enable MMR diversity
    top_k=3
)
print("\n✅ RAG Agent initialized!")


## Step 4: Load Documents


In [ ]:
# Check documents folder
documents_path = "docs"
docs_folder = Path(documents_path)

if not docs_folder.exists():
    print(f"📁 Creating documents folder: {documents_path}")
    docs_folder.mkdir(exist_ok=True)

# List available documents
supported_extensions = [".pdf", ".txt", ".md"]
documents = []
for ext in supported_extensions:
    documents.extend(list(docs_folder.glob(f"*{ext}")))
    documents.extend(list(docs_folder.glob(f"**/*{ext}")))

if documents:
    print(f"📚 Found {len(documents)} document(s):")
    for doc in documents:
        print(f"   - {doc.name}")
else:
    print(f"⚠️  No documents found in '{documents_path}/'")
    print(f"   The test document should be at: {documents_path}/test_document.txt")


In [ ]:
# Load documents into the vector store
print("\n" + "="*60)
print("📥 Loading Documents into Vector Store")
print("="*60)

agent.load_documents(documents_path)

print("\n✅ Document loading complete!")


## Step 5: Test Queries


In [ ]:
# Test query 1 - Note: query() now returns a dict
question1 = "What is RAG?"
print(f"\n{'='*60}")
print(f"❓ Question: {question1}")
print(f"{'='*60}")

result1 = agent.query(question1)
print(f"\n💬 Answer:\n{result1['answer']}")
if result1.get('sources'):
    print(f"\n📚 Sources: {', '.join(result1['sources'])}")


In [ ]:
# Test query 2 - With conversation memory
question2 = "What technologies are used in this project?"
print(f"\n{'='*60}")
print(f"❓ Question: {question2}")
print(f"{'='*60}")

result2 = agent.query(question2)
print(f"\n💬 Answer:\n{result2['answer']}")
if result2.get('sources'):
    print(f"\n📚 Sources: {', '.join(result2['sources'])}")


In [ ]:
# Test query 3
question3 = "Explain how vector databases work"
print(f"\n{'='*60}")
print(f"❓ Question: {question3}")
print(f"{'='*60}")

result3 = agent.query(question3)
print(f"\n💬 Answer:\n{result3['answer']}")
if result3.get('sources'):
    print(f"\n📚 Sources: {', '.join(result3['sources'])}")


## Step 6: Multiple Questions Session


In [ ]:
# Run multiple questions in a session (with conversation memory)
questions = [
    "What is the main purpose of RAG?",
    "What are the key components of RAG?",
    "What are the benefits of using RAG?",
    "What is a vector database?"
]

print("\n" + "="*60)
print("🤖 Q&A Session (with Conversation Memory)")
print("="*60)
print("Note: Each question can use context from previous questions!\n")

for i, question in enumerate(questions, 1):
    print(f"\n{'─'*60}")
    print(f"Question {i}/{len(questions)}: {question}")
    print(f"{'─'*60}")
    
    result = agent.query(question)
    print(f"\nAnswer:\n{result['answer']}")
    if result.get('sources'):
        print(f"\n📚 Sources: {', '.join(result['sources'])}")
    print("\n" + "─"*60)


## Step 7: Interactive Query Function


In [ ]:
def ask_question(question: str) -> dict:
    """Helper function to ask questions"""
    print(f"\n{'='*60}")
    print(f"❓ Your Question: {question}")
    print(f"{'='*60}")
    
    result = agent.query(question)
    print(f"\n💬 Answer:\n{result['answer']}")
    if result.get('sources'):
        print(f"\n📚 Sources: {', '.join(result['sources'])}")
    
    return result

# Example usage
ask_question("What are the use cases of RAG?")


## Step 8: Explore Vector Store


In [ ]:
# Check vector store information
if agent.vectorstore:
    print("📊 Vector Store Information:")
    print(f"   Location: {agent.persist_directory}")
    
    # Get collection info
    collection = agent.vectorstore._collection
    if collection:
        count = collection.count()
        print(f"   Total chunks stored: {count}")
        
        # Sample a few documents
        if count > 0:
            print(f"\n   Sample stored chunks:")
            results = collection.peek(limit=min(3, count))
            if 'metadatas' in results:
                for i, metadata in enumerate(results['metadatas'][:3], 1):
                    source = metadata.get('source', 'Unknown')
                    filename = Path(source).name if source else 'Unknown'
                    print(f"      {i}. {filename}")
else:
    print("⚠️  Vector store not initialized")


## Step 9: Test Similarity Search


In [ ]:
# Test direct similarity search
if agent.vectorstore:
    test_query = "machine learning"
    print(f"🔍 Testing similarity search for: '{test_query}'")
    print(f"{'='*60}")
    
    # Get retriever
    retriever = agent.vectorstore.as_retriever(search_kwargs={"k": 3})
    
    # Retrieve similar documents
    docs = retriever.get_relevant_documents(test_query)
    
    print(f"\n📄 Found {len(docs)} relevant chunks:\n")
    
    for i, doc in enumerate(docs, 1):
        print(f"{'─'*60}")
        print(f"Chunk {i}:")
        print(f"Source: {Path(doc.metadata.get('source', 'Unknown')).name}")
        print(f"Content (first 200 chars): {doc.page_content[:200]}...")
        print()
else:
    print("⚠️  Vector store not initialized")


## Step 10: Clear History and Start Fresh


In [ ]:
# Clear conversation history
print("🧹 Clearing conversation history...")
if agent.memory:
    agent.clear_history()
    print("✅ History cleared!")
    print(f"   Previous exchanges: {agent.memory.get_exchange_count()}")
else:
    print("⚠️  Memory not enabled")

# Test a fresh question
fresh_question = "What is the first thing mentioned about RAG?"
ask_question(fresh_question)


## Step 11: Custom Questions

Try your own questions below!


In [ ]:
# YOUR CUSTOM QUESTION HERE
your_question = "What are the benefits of using vector databases?"

ask_question(your_question)


## Step 12: Advanced Features Demo

### 12.1: Conversation Memory


In [ ]:
# Demonstrate conversation memory
if agent.memory:
    print("\n" + "="*60)
    print("💾 Conversation Memory Demo")
    print("="*60)
    
    # Show conversation history
    messages = agent.memory.get_all_messages()
    print(f"\n📝 Total messages in history: {len(messages)}")
    print(f"🔄 Number of exchanges: {agent.memory.get_exchange_count()}")
    
    if messages:
        print("\n📜 Recent conversation:")
        for msg in messages[-4:]:  # Show last 4 messages
            role_icon = "👤" if msg.role == "user" else "🤖"
            print(f"   {role_icon} {msg.role}: {msg.content[:100]}...")
    
    # Show conversation context
    context = agent.memory.get_conversation_context(include_recent=2)
    if context:
        print("\n📋 Conversation Context (for next query):")
        print(context[:300] + "..." if len(context) > 300 else context)
else:
    print("⚠️  Memory not enabled. Enable with enable_memory=True")


### 12.2: Advanced Retrieval (Optional)


In [ ]:
# Demonstrate advanced retrieval features
print("\n" + "="*60)
print("🎯 Advanced Retrieval Features")
print("="*60)

print("\nAvailable features:")
print("   1. Reranking: Cross-encoder reranking for better relevance")
print("   2. Hybrid Search: Combines semantic + keyword (BM25) search")
print("   3. MMR: Maximal Marginal Relevance for diverse results")

# Example: Initialize agent with advanced features
print("\n💡 To enable advanced features, initialize agent like this:")
print("""
agent_advanced = RAGAgent(
    model_name='llama2',
    enable_memory=True,
    enable_reranking=True,    # Enable reranking
    enable_hybrid=True,       # Enable hybrid search
    enable_mmr=True,          # Enable MMR diversity
    top_k=5
)
""")

# Check if advanced retriever is available
if hasattr(agent, 'advanced_retriever') and agent.advanced_retriever:
    print("✅ Advanced retriever is active!")
    print(f"   - Reranking: {agent.advanced_retriever.enable_reranking}")
    print(f"   - Hybrid: {agent.advanced_retriever.enable_hybrid}")
    print(f"   - MMR: {agent.advanced_retriever.enable_mmr}")
else:
    print("ℹ️  Advanced retriever not enabled. Use enable_reranking/hybrid/mmr=True")


### 12.3: Summary and Next Steps


In [ ]:
print("\n" + "="*60)
print("✅ End-to-End Process Complete!")
print("="*60)
print("\n📋 Summary:")
print("   1. ✅ RAG Agent initialized with advanced features")
print("   2. ✅ Documents loaded and processed (with automatic OCR)")
print("   3. ✅ Vector store created")
print("   4. ✅ Queries tested with conversation memory")
print("   5. ✅ Source citations included")
print("\n🚀 Next Steps:")
print("   • Try the Web UI: streamlit run src/web_ui.py")
print("   • Use the REST API: uvicorn src.api:app --reload")
print("   • Enable advanced retrieval (reranking, hybrid, MMR)")
print("   • Add more documents to the 'docs/' folder")
print("   • Experiment with different models: ollama pull mistral")
print("   • Try multi-turn conversations")
print("\n💡 Tips:")
print("   • Vector store is saved in 'vectorstore/' folder")
print("   • You can reload it without re-ingesting")
print("   • Conversation memory helps with follow-up questions")
print("   • Use specific questions for better results")
print("   • Check sources to verify answers")
print("\n🌐 Web Interfaces:")
print("   • Streamlit UI: rag-ui or streamlit run src/web_ui.py")
print("   • REST API: rag-api or uvicorn src.api:app --reload")
print("   • API Docs: http://localhost:8000/docs")
print("\n" + "="*60)
